# 02a - Chunk and embed the filings (text version)

Steps: take the extracted sections from notebook 01, cut them into overlapping chunks of about 1600 characters,
measure the real token length of every chunk, embed them with `BAAI/bge-m3`, run a small smoke test, and upload
chunks, vectors and a short report to the private dataset (folder `text/`).

In this version tables are plain text, exactly as extracted. The table-aware version is notebook `02b`;
a later notebook compares the two.

**Before running:** Runtime > T4 GPU, and the Colab Secrets `HF_TOKEN` (write access) and, only while the
repository is private, `GITHUB_TOKEN` (read-only). About 10 minutes.

In [ ]:
import os
import subprocess
import sys

from google.colab import userdata

REPO = "10K-agentic-rag"
BRANCH = "main"
try:  # only needed while the repository is private (a read-only token for this one repository)
    token = userdata.get("GITHUB_TOKEN")
except Exception:  # noqa: BLE001  the secret does not exist: the repository is public
    token = None
login = f"{token}@" if token else ""
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "-b", BRANCH, f"https://{login}github.com/ilaria-cesa/{REPO}.git"],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
os.chdir(REPO)
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements.txt

In [ ]:
import random
from pathlib import Path

import numpy as np
import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(torch.cuda.get_device_name(0))

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # never hard-code the token
HF_REPO = "IlariaCesa/10k-agentic-rag-data"

Get the extracted sections from the dataset and cut them into chunks (CPU, no model needed).
Size 1600 characters, overlap 200, paragraphs are not split; every chunk also gets a context header
(company, period, section) that is used only for the embedding.

In [ ]:
from src.chunking.run import main as make_chunks
from src.hub.dataset import download_folder

download_folder(HF_REPO, "parsed", Path("data"))
make_chunks(["--out", "data"])

In [ ]:
from src.retrieval.embed import load_chunks

chunks = load_chunks("data/chunks_1600.jsonl")
texts = [c["embed_text"] for c in chunks]
print(len(chunks), "chunks")

Real token length of every chunk (the chunker only estimates it). The model limit is set above the longest
chunk, so nothing is silently cut.

In [ ]:
from sentence_transformers import SentenceTransformer

from src.retrieval.embed import length_report, token_lengths

MODEL_NAME = "BAAI/bge-m3"
MAX_LEN = 640
model = SentenceTransformer(MODEL_NAME, device="cuda", model_kwargs={"dtype": torch.float16})
model.max_seq_length = MAX_LEN

lengths = token_lengths(texts, model.tokenizer)
report = length_report(lengths, MAX_LEN)
report["chars_per_token"] = round(sum(len(t) for t in texts) / sum(lengths), 2)
assert report["n_over_limit"] == 0, "some chunks would be truncated: raise MAX_LEN"
print(report)

In [ ]:
from src.retrieval.embed import embed_texts

embeddings = embed_texts(model, texts, batch_size=32)
print(embeddings.shape, embeddings.dtype)

Smoke test (not an evaluation): do three simple questions find sensible passages?

In [ ]:
from src.retrieval.index import build_index, search

index = build_index(embeddings)
for q in [
    "What are the main credit risks for JPMorgan Chase?",
    "How does Apple describe supply chain risks?",
    "How do catastrophe losses affect Travelers?",
]:
    scores, ids = search(index, embed_texts(model, [q], progress=False), k=3)
    print("\nQ:", q)
    for s, i in zip(scores[0], ids[0]):
        print(f"  {s:.3f}  {chunks[i]['chunk_id']}  {chunks[i]['text'][:110]!r}")

Save everything the next notebooks need in the `text/` folder of the private dataset.

In [ ]:
import importlib.metadata as md
import json

from src.hub.dataset import upload_folder

out = Path("out_text")
out.mkdir(exist_ok=True)
report["model"] = MODEL_NAME
report["versions"] = {
    p: md.version(p)
    for p in [
        "sentence-transformers",
        "transformers",
        "torch",
        "faiss-cpu",
        "numpy",
        "huggingface_hub",
    ]
}
report["n_chunks"] = len(chunks)
np.save(out / "embeddings_bge-m3.npy", embeddings.astype(np.float16))
(out / "chunks_1600.jsonl").write_text(
    Path("data/chunks_1600.jsonl").read_text(encoding="utf-8"), encoding="utf-8"
)
(out / "embedding_report.json").write_text(json.dumps(report, indent=2))
(out / "meta.json").write_text(
    json.dumps(
        {
            "variant": "text",
            "model": MODEL_NAME,
            "max_seq_length": MAX_LEN,
            "n": len(chunks),
            "dim": int(embeddings.shape[1]),
            "dtype": "float16",
            "normalized": True,
            "first_chunk_ids": [c["chunk_id"] for c in chunks[:3]],
        },
        indent=2,
    )
)
print(upload_folder(HF_REPO, out, "text"))
print(json.dumps(report, indent=2))

Next: `02b_chunk_and_embed_tables.ipynb` (tables as structured text) and then the comparison of the two versions.
Copy the printed report into the chat if you want it saved in `results/`.